In [ ]:
import numpy as np
from collections import defaultdict
from itertools import combinations
from net import *
from tensorflow.keras.models import load_model
import tensorflow as tf
import time
import concurrent.futures
import os

gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        # only use GPU 1
        tf.config.experimental.set_visible_devices(gpus[0], 'GPU')
        tf.config.experimental.set_memory_growth(gpus[0], True)  #
    except RuntimeError as e:
        print(e)
        
NET_trained = make_resnet_ghor(depth=1, reg_param=10**-5)
NET_trained.load_weights("./net8_small.h5")
rng = np.random.default_rng()
M = 10000000 # the data complexity, we use 50000000 for getting precise results; 
             # while setting N=10000000 also can get a good result in this case. 
N = 64       # the oracle input length
    
def boolean_oracle(x):
    """
    call the Boolean oracle and return the value f(x)
    """
    rlist = NET_trained.predict(x, batch_size= 50000, verbose=0)
    res = []
    # encoding into -1, 1
    for r in rlist:
        if(r > 0.5):
            res.append(1)
        else:
            res.append(0)
    return np.array(res).reshape(len(x), 1)
    # return f(x)

# def boolean_oracle(x):
#     """
#     test
#     """
#     t = x[:, 0].reshape(-1, 1)
#     return t
#     # return f(x)

def hardcode_samples_i(i):
    # return 32 (for len(Z) = 0,1,...,31): Y, f(Y), Y_Z, f(Y'_Z)
    print("data generated now ...")

    Y_list = []
    Ydot_list = []
    F_YZ_list = []
    F_YdotZ_list = []
    
    Z = np.random.randint(0, 2, size=(M, N - i))
    Y = np.random.randint(0, 2, size=(M, i))
    Ydot = np.random.randint(0, 2, size=(M, i))
    
    Y_Z = np.concatenate((Y, Z), axis=1)
    Ydot_Z = np.concatenate((Ydot, Z), axis=1)
    
    f_y_z = (-1)**boolean_oracle(Y_Z)
    f_ydot_z = (-1)**boolean_oracle(Ydot_Z)

    print(str(i) + "-th loop datageneration completed!")
    return Y, Ydot, f_y_z, f_ydot_z


2025-10-09 22:08:14.085499: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-09 22:08:14.157828: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760018894.246429 3470655 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760018894.276810 3470655 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-10-09 22:08:14.363060: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [ ]:
def sample_inputs(n, m):
    """
    random sample generation
    """
    return np.random.randint(0, 2, size=(m, n))

def fyz_times_chi(y, z,S, X):
    res = 1
    for s in S:
        res *= X[s]
    return res

def sub_estimator(S, J, Y, Ydot, f_y_z, f_ydot_z):
    start_time = time.time()
    N = 32

    # calculate the chi_y, chi_y_dot 
    chi_y = (-1)**(np.sum(Y[:, S], axis=1, keepdims=True))
    chi_y_dot = (-1)**(np.sum(Ydot[:, S], axis=1, keepdims=True))
    
    # calculate res
    res = f_y_z * f_ydot_z
    res *= chi_y
    res *= chi_y_dot

    # cal size of res == 1
    x = np.sum(res == 1)
    
    # get the final results
    e = (2 * x - M) / (M)
    
    end_time = time.time()
    execution_time = end_time - start_time
    # print(f"exc-time: {execution_time} s")
    
    return e

def Goldreich_Levin(tau, N=64):
    backup_path = f"./backup/{M}_{int(tau*1000)}_modelgohr8_parallel2.txt"
    current_B = [([], 0)]
    next_B = []

    def process_B(B):
        """Processes a single B and returns valid splits."""
        valid_Bs = []
        B1 = (B[0], B[1] + 1)
        B2 = (B[0] + [B[1]], B[1] + 1)
        
        weight_1 = sub_estimator(B[0], B[1] + 1, Y_ls, Ydot_ls, f_y_z_ls, f_ydot_z_ls)
        weight_2 = sub_estimator(B[0] + [B[1]], B[1] + 1, Y_ls, Ydot_ls, f_y_z_ls, f_ydot_z_ls)

        if weight_1 > tau**2 / 2.0:
            valid_Bs.append(B1)
        if weight_2 > tau**2 / 2.0:
            valid_Bs.append(B2)
        return valid_Bs
    
    # omit -> accelerate the progress
    for i in range(7):
        while(current_B):
            B = current_B[0]

            B1 = (B[0], B[1] + 1)
            B2 = (B[0] + [B[1]], B[1] + 1)
            next_B.append(B1)
            next_B.append(B2)
            current_B.remove(B)
        current_B = next_B[:]
        next_B = []
        print("current B numbers: " + str(len(current_B)))

    for i in range(7, 64):
        # public data
        Y_ls, Ydot_ls, f_y_z_ls, f_ydot_z_ls = hardcode_samples_i(i+1)
        
        # Parallel processing of current_B
        next_B = []
        with concurrent.futures.ThreadPoolExecutor() as executor:
            futures = {executor.submit(process_B, B): B for B in current_B}
            for future in concurrent.futures.as_completed(futures):
                next_B.extend(future.result())

        current_B = next_B[:]
        next_B = []

        with open(backup_path, "w") as file:
            file.write(",".join(map(str, current_B)))
        print("current B numbers: " + str(len(current_B)))
    return current_B

# precompute the data, used in recovery the coeff
_data_used_for_get_fourier_coeff = np.random.randint(0, 2, size=(1000000, 64))
_f_get_coeff = (-1)**boolean_oracle(_data_used_for_get_fourier_coeff)

def get_Fourier_coeff(S_list):
    res = []
    for s in S_list:
        chi_x = (-1)**(np.sum(_data_used_for_get_fourier_coeff[:, s[0]], axis=1, keepdims=True))
        t = _f_get_coeff * chi_x

        one = np.sum(t == 1) 
        e = (2*one - 1000000)/(1000000*1.0)
        res.append(e)
    
    return res

def reconstruct_bf(S_list, coeff_list, tau, store=True):
    _M = 10000000
    x = np.random.randint(0, 2, size=(_M, 64))
    y = (-1)**boolean_oracle(x)  # ND oracle
    item_num = 0
    estimator_x = np.zeros((_M, 1))
    for i, s in enumerate(S_list):
        if(abs(coeff_list[i]) > (tau / 2.0)):
            estimator_x += coeff_list[i] * ((-1)**(np.sum(x[:, s[0]], axis=1, keepdims=True)))
            item_num += 1
    res = []
    for r in estimator_x:
        if(r[0] > 0):
            res.append(1)
        else:
            res.append(-1)
    res = np.array(res).reshape(_M, 1)

    diff_pos_ML = np.sum((y == 1) & (res == -1)) /np.sum((y == 1))
    diff_neg_ML = np.sum((y == -1) & (res == 1)) / np.sum((y == -1))
    diff_ML = np.sum(y != res) / _M
    print(diff_ML)
    print(diff_pos_ML)
    print(diff_neg_ML)

    
    # test real sample from speck
    print("now test for real data")
    import speck as sp
    X, Y = sp.make_train_data(_M, 8, (0x0040, 0x0000)) # the first var:  data size
                                                       # the second var: round number
                                                       # the third var:  input diff
    Y = Y.reshape(-1,1)
    Y = (-1)**Y.astype(np.int16)

    print("generation encryption data completed! ")
    estimator_x = np.zeros((_M, 1))
    for i, s in enumerate(S_list):
        if(abs(coeff_list[i]) > (tau / 2.0)):
            _temp_sum = np.sum(X[:, s[0]], axis=1, keepdims=True, dtype=np.int64)
            estimator_x += coeff_list[i] * ((-1) ** _temp_sum)
    res = []
    for r in estimator_x:
        if(r[0] > 0):
            res.append(-1)
        else:
            res.append(1)
            
    print(estimator_x[:4])
    print(np.sum((Y == 1)))
    print(np.sum((Y == -1)))

    res = np.array(res).reshape(_M, 1)
    diff_pos_real = np.sum((Y == 1) & (res == -1)) /np.sum((Y == 1))
    diff_neg_real = np.sum((Y == -1) & (res == 1)) / np.sum((Y == -1))
    diff_real = np.sum(Y != res) / _M
    print("ACC" + str(diff_real))
    print("TPR" + str(diff_pos_real))
    print("TNR" + str(diff_neg_real))

    
    # store the data
    params = {
    "Query times": M,
    "tau": tau,
    "terms": item_num,
        
    "diff_ML": diff_ML,
    "diff_pos_ML": diff_pos_ML,
    "diff_neg_ML": diff_neg_ML,
        
    "acc_data": diff_real,
    "TPR_data": diff_pos_real,
    "TNP_data": diff_neg_real
    }
    path_file = f"./data_Fourier/Parallel{M}_{int(tau*1000)}gohr7_model3.txt"
    with open(path_file, "w") as file:
        for param_name, param_value in params.items():
            file.write(f"{param_name}: {param_value}\n")
        file.write(" ".join(map(str, S_list)) + "\n")
        file.write(" ".join(map(str, coeff_list)) + "\n")


    


I0000 00:00:1760018914.029808 3470958 service.cc:148] XLA service 0x7f6a80015e00 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1760018914.029869 3470958 service.cc:156]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
2025-10-09 22:08:34.069287: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1760018914.473084 3470958 cuda_dnn.cc:529] Loaded cuDNN version 91200
2025-10-09 22:08:36.054755: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:306] Allocator (GPU_0_bfc) ran out of memory trying to allocate 12.99GiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were available.
I0000 00:00:1760018917.351828 3470958 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the l

In [3]:
tau = 0.1
B = Goldreich_Levin(tau)
print("done")
print(B)
coefflist = get_Fourier_coeff(B)
print(coefflist)
reconstruct_bf(B, coefflist, tau)
print(len(coefflist))

current B numbers: 2
current B numbers: 4
current B numbers: 8
current B numbers: 16
current B numbers: 32
current B numbers: 64
current B numbers: 128
data generated now ...
8-th loop datageneration completed!
current B numbers: 16
data generated now ...
9-th loop datageneration completed!
current B numbers: 24
data generated now ...
10-th loop datageneration completed!
current B numbers: 27
data generated now ...
11-th loop datageneration completed!
current B numbers: 25
data generated now ...
12-th loop datageneration completed!
current B numbers: 26
data generated now ...
13-th loop datageneration completed!
current B numbers: 19
data generated now ...
14-th loop datageneration completed!
current B numbers: 15
data generated now ...
15-th loop datageneration completed!
current B numbers: 14
data generated now ...
16-th loop datageneration completed!
current B numbers: 14
data generated now ...
17-th loop datageneration completed!
current B numbers: 14
data generated now ...
18-th l